# Test your own banana leaf photo
This short notebook loads the saved model without retraining. Select **Banana Leaf (PyTorch)**, choose **Run All**, then use **Upload leaf photo**. Keep this notebook beside the main assignment notebook and its `output` folder.

The model chooses among cordana, healthy, pestalotiopsis and sigatoka. Its probabilities are uncalibrated; unrelated images and other diseases are outside the four-class task. No deployment has been started.

### 1. Import tools

In [1]:
from pathlib import Path
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torchvision import models, transforms
from IPython.display import display
import ipywidgets as widgets
torch.set_num_threads(4)
print("Saved-model testing tools are ready.")

Saved-model testing tools are ready.


### 2. Find the saved model

In [2]:
output_folder = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("output")
model_file = output_folder / "banana_leaf_disease_model.pt"
if not model_file.exists():
    raise FileNotFoundError("Run the main assignment notebook to save the model first.")
print("Loading:", model_file.resolve())

Loading: F:\janv\output\banana_leaf_disease_model.pt


### 3. Reload the model and preprocessing

In [3]:
saved_checkpoint = torch.load(model_file, map_location="cpu", weights_only=True)
saved_model = models.mobilenet_v3_small(weights=None)
saved_model.classifier[-1] = nn.Linear(saved_model.classifier[-1].in_features, 4)
saved_model.load_state_dict(saved_checkpoint["model_state"])
saved_model.eval()
saved_transform = transforms.Compose([
    transforms.Resize(saved_checkpoint["resize_shorter_side"]),
    transforms.CenterCrop(saved_checkpoint["crop_size"]), transforms.ToTensor(),
    transforms.Normalize(saved_checkpoint["mean"], saved_checkpoint["std"])
])
print("Reloaded. Classes:", saved_checkpoint["class_names"])

Reloaded. Classes: ['cordana', 'healthy', 'pestalotiopsis', 'sigatoka']


### 4. Define one-photo prediction

In [4]:
def predict_leaf_photo(photo):
    input_tensor = saved_transform(photo.convert("RGB")).unsqueeze(0)
    with torch.inference_mode():
        probabilities = saved_model(input_tensor).softmax(1).numpy()[0]
    predicted_number = int(probabilities.argmax())
    return saved_checkpoint["class_names"][predicted_number], probabilities

print("Single-photo prediction ready using the saved model.")

Single-photo prediction ready using the saved model.


### 5. Define the probability chart

In [5]:
def show_leaf_prediction(photo, title="Your leaf photo"):
    predicted_class, probabilities = predict_leaf_photo(photo)
    figure, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(photo.convert("RGB"))
    axes[0].set_title(title)
    axes[0].axis("off")
    bars = axes[1].barh(saved_checkpoint["class_names"], probabilities, color="#438c56")
    axes[1].bar_label(bars, labels=[f"{value:.1%}" for value in probabilities], padding=3)
    axes[1].set(xlim=(0, 1.15), xlabel="Model probability", title=f"Prediction: {predicted_class}")
    figure.tight_layout()
    plt.show()
    print(f"Predicted class: {predicted_class}; top score: {probabilities.max():.1%}")
    return predicted_class, probabilities

print("Photo and probability-chart display ready.")

Photo and probability-chart display ready.


### 6. Define the upload handler

In [6]:
import ipywidgets as widgets
upload_result = widgets.Output()

def handle_leaf_upload(change):
    with upload_result:
        upload_result.clear_output(wait=True)
        if not upload_button.value:
            return
        files = upload_button.value
        uploaded_file = next(iter(files.values())) if isinstance(files, dict) else files[0]
        try:
            with Image.open(io.BytesIO(bytes(uploaded_file["content"]))) as image:
                photo = image.convert("RGB")
            show_leaf_prediction(photo, uploaded_file.get("name", "Uploaded photo"))
        except (OSError, ValueError) as error:
            print("Choose a readable JPG, PNG or WebP leaf photo.", str(error))

print("Upload handler ready.")

Upload handler ready.


### 7. Upload your own image

In [7]:
upload_button = widgets.FileUpload(accept="image/*", multiple=False, description="Upload leaf photo")
upload_button.observe(handle_leaf_upload, names="value")
display(widgets.VBox([widgets.HTML("<b>Test your own banana leaf photo</b>"), upload_button, upload_result]))
print("Ready: upload a photo to test the saved model. Deployment has not started.")

Ready: upload a photo to test the saved model. Deployment has not started.
